### Q1. Preparation

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1. How many lesson pages

In [4]:
len(documents)

72

### Q2. Indexing and searching

What's the filename of the first result?

In [5]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [6]:
from minsearch import Index

index = Index(
    text_fields=["filename", "content"], # text fields that you can use to perform the search
    keyword_fields=["filename"] # keyword fields are used to filter (exact match for) - select * from index where course = 'llm-zoomcamp'
)

index.fit(documents)

In [7]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [8]:
index.search(
    question,
    num_results=1
)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### Q3. RAG

How many input (prompt) tokens did we send to the model for this request?

In [9]:
import sys

from pathlib import Path

# workaround because the rag_helper is in the parent folder
sys.path.append(str(Path.cwd().parent))

from rag_helper import RAGBase

In [10]:
class RAGLessons(RAGBase): # RAGBase already uses model="gpt-5.4-mini"
    def search(self, query, num_results=5):
        boost_dict = {"content": 5.0, "filename": 1.0}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["filename"])
            lines.append("Content: " + doc["content"])
            lines.append("")

        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage.input_tokens

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, input_tokens = self.llm(prompt)
        return {
            'answer': answer,
            'input_tokens':input_tokens
        }

In [11]:
rag_lessons = RAGLessons(index, openai_client)

In [12]:
rag_lessons.rag('How does the agentic loop keep calling the model until it stops?')

{'answer': 'The loop keeps calling the model with a `while True` loop, and after each response it checks whether the model returned any `function_call` items.\n\n- If there are function calls, the code runs the tools, appends the tool outputs to `messages`, and loops again.\n- If there are no function calls, it `break`s out of the loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: it keeps going until the model returns a final message with no more tool calls.',
 'input_tokens': 7126}

In [13]:
response_dict = rag_lessons.rag('How does the agentic loop keep calling the model until it stops?')
q3_input_tokens = response_dict['input_tokens']

### Q4. Chunking

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

### Q5. RAG with chunking

In [16]:
index_chunk = Index(
    text_fields=["filename", "content"], # text fields that you can use to perform the search
    keyword_fields=["filename"] # keyword fields are used to filter (exact match for) - select * from index where course = 'llm-zoomcamp'
)

index_chunk.fit(documents)

In [17]:
rag_lessons_chunked = RAGLessons(index_chunk, openai_client)

In [18]:
rag_lessons_chunked.rag('How does the agentic loop keep calling the model until it stops?')

{'answer': 'It keeps a `while True` loop that:\n\n1. calls the model with the full `messages` history,\n2. checks the response for any `function_call` items,\n3. runs those tools and appends the tool outputs to `messages`,\n4. repeats if there was at least one function call.\n\nIt stops when the model returns a response with **no function calls**:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the model decides when to stop, and the loop exits when it answers directly instead of asking for another tool call.',
 'input_tokens': 7126}

In [19]:
rag_lessons_chunked.rag('How does the agentic loop keep calling the model until it stops?')

{'answer': 'It keeps calling the model in a `while True` loop and checks whether the response contains any `function_call` items.\n\n- If there is a function call, it runs the tool, appends the tool output to `messages`, and loops again.\n- If there are no function calls, it breaks out of the loop.\n\nSo the stop condition is: **no function calls in the model’s response**.',
 'input_tokens': 7126}

In [20]:
response_chunked_dict = rag_lessons_chunked.rag('How does the agentic loop keep calling the model until it stops?')
q4_input_tokens = response_chunked_dict['input_tokens']

In [21]:
q4_input_tokens/q3_input_tokens

1.0

### Q6. Turning it into an agent

In [22]:
INSTRUCTIONS = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [25]:
# from https://github.com/alexeygrigorev/toyaikit README.md:

from openai import OpenAI

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from typing import Any

# Create tools
tools = Tools()

def search(query, num_results: int =5) -> list[dict[str, Any]]:
    """

    Search the chunk index for relevant documents. The documents are 
    leassons from the LLM Zoomcamp course, available in GitHub. In each document,
    there is a filename, a start and a content.

    Args:

        query: User search query.
        num_results: Maximum number of results to return.

    """
        
    boost_dict = {"content": 5.0, "filename": 1.0}

    return index_chunk.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
    )

tools.add_tool(search)

# Create chat interface and client
chat_interface = IPythonChatInterface()
openai_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

# Create and run chat assistant
runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=openai_client
)

runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user